In [2]:
import os
from io import StringIO
import pandas as pd
!pip install unidecode
import unidecode
import re
from datetime import datetime
import unicodedata
from google.colab import files

numero_lote = int(input("Qual o número do lote? "))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 4.7 MB/s eta 0:00:00
Qual o número do lote? 56


In [3]:
# Exibir todas as colunas
pd.set_option('display.max_columns', None)

# ====== Upload do arquivo ======
print("Faça upload do arquivo 'consolidado normal.xlsx'")
uploaded = files.upload()   # abre seletor de arquivos no Colab

# Como o usuário vai subir, o nome do arquivo é o mesmo
file_path = "consolidado normal.xlsx"

# ====== Configuração das colunas que devem ser texto ======
colunas_cpfs = ['CPF'] + [f'GF_CPF_MEMBRO_0{i}' for i in range(1, 10)] + ['GF_CPF_MEMBRO_10']
colunas_forcar_texto = colunas_cpfs + ['NIS', 'LATITUDE', 'LONGITUDE', 'NOME', 'ENDERECO','ITEM']

dtypes_dict = {col: str for col in colunas_forcar_texto}

# ====== Leitura do Excel ======
df = pd.read_excel(
    file_path,
    dtype=dtypes_dict,       # força essas colunas a texto
    keep_default_na=False,   # '' continua string, não vira NaN
    engine="openpyxl"
)

# ====== Limpeza de quebras de linha ======
for col in df.select_dtypes(include='object'):
    df[col] = (
        df[col]
        .str.replace('\r', ' ', regex=False)
        .str.replace('\n', ' ', regex=False)
        .str.strip()
    )

df1 = df

# Mostrar primeiras linhas para conferência
df1.head()


Faça upload do arquivo 'consolidado normal.xlsx'


Saving consolidado normal.xlsx to consolidado normal.xlsx


,ITEM,NOME,CPF,NIS,ENDERECO,BAIRRO,LATITUDE,LONGITUDE,SITUACAO DA HABITACAO,CONCLUSAO DO LAUDO TECNICO DE ENGENHARIA,NATUREZA,MUNICIPIO
0,1,YASMIN REZENDE,012 63828833,,TRECHO VITOR MOREIRA,,29.9324324353208°,52.719454353084°,I,I,urbano,sao martinho da serra
1,2,ALICE GONÇALVES,962.950.124-42,,,beira linha,,,I,I,urbano,sao martinho da serra
2,3,Agatha Ferreira,,,,NOSSA SENHORA DA CONCEIÇÃO,,,I,I,urbano,sao martinho da serra
3,4,Carlos Eduardo Costa,61092477829dfdaf,,,são cristóvão,,,I,I,urbano,sao martinho da serra
4,5,Dr. Vitor Hugo Oliveira,735.934.596-04,,PRAIA DE CARVALHO,JARAGUÁ,,,I,I,urbano,SÃO JERÔNIMO


In [4]:
# Substituir todos os NaN do DataFrame por string vazia
df = df.fillna("")

# Verificar o resultado
df.tail()

,ITEM,NOME,CPF,NIS,ENDERECO,BAIRRO,LATITUDE,LONGITUDE,SITUACAO DA HABITACAO,CONCLUSAO DO LAUDO TECNICO DE ENGENHARIA,NATUREZA,MUNICIPIO
376,22,Bárbara Duarte,57146187691,,MORRO SOUZA,VILA SANTA MONICA 1ª SEÇÃO,,,A ser removida para viabilizar soluções urbana...,HABITÁVEL ADJACENTE - A ser removida para via...,urbano,canoas
377,29,Igor Barbosa,10799910856,,ladeira de rocha,santa efigênia,,,A ser removida para viabilizar soluções urbana...,HABITÁVEL ADJACENTE - A ser removida para via...,urbano,canoas
378,30,Eloah Costa,99984667847,,VEREDA ANTÔNIO NOVAES,NOSSA SENHORA DE FÁTIMA,,,A ser removida para viabilizar soluções urbana...,HABITÁVEL ADJACENTE - A ser removida para via...,urbano,canoas
379,32,Pedro Lucas Castro,89055974676,,fazenda duarte,sagrada família,,,A ser removida para viabilizar soluções urbana...,HABITÁVEL ADJACENTE - A ser removida para via...,urbano,canoas
380,34,Enzo Gabriel Pereira,70093289430,,VIADUTO DE ALVES,VILA SUZANA SEGUNDA SEÇÃO,,,A ser removida para viabilizar soluções urbana...,HABITÁVEL ADJACENTE - A ser removida para via...,urbano,canoas


In [6]:
# Lista das colunas a serem tratadas normalmente
colunas_tratadas = ["NOME", "ENDERECO", "BAIRRO", "LATITUDE", "LONGITUDE", "MUNICIPIO", "NATUREZA"]

# Função para normalizar os textos (exceto CPF e NIS)
def normalizar_texto(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).upper()
    texto = unidecode.unidecode(texto)
    texto = re.sub(r"[^A-Z0-9]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

# Função específica para CPF e NIS (mantém apenas números, inclusive zeros à esquerda)
def limpar_documento(texto):
    if pd.isna(texto):
        return "00000000000"  # Se for NaN, devolve 11 zeros
    # Remove tudo que não for número
    texto = re.sub(r"[^0-9]", "", str(texto))
    return texto.zfill(11)  # Mesmo que vazio, vai virar 11 zeros


# Aplicar formatação
for coluna in colunas_tratadas:
    df[coluna] = df[coluna].apply(normalizar_texto)

# Aplicar limpeza especial para CPF e NIS
df["CPF"] = df["CPF"].apply(limpar_documento)
df["NIS"] = df["NIS"].apply(limpar_documento)


In [7]:
# Remover espaços das colunas CPF e NIS, latitude, longitude e município
df["CPF"] = df["CPF"].str.replace(" ", "", regex=True)
df["NIS"] = df["NIS"].str.replace(" ", "", regex=True)
df["LATITUDE"] = df["LATITUDE"].str.replace(" ", "", regex=True)
df["LONGITUDE"] = df["LONGITUDE"].str.replace(" ", "", regex=True)
df["MUNICIPIO"] = df["MUNICIPIO"].str.replace(" ", "", regex=True)




In [8]:
# Converter todos os valores da coluna para minúsculas antes de substituir
df["NATUREZA"] = df["NATUREZA"].str.lower().replace({
    "RURAL": "R",
    "URBANA": "U",
    "rural":"R",
    "urbana":"U",
    "urbano":"U",
    "URBANO":"U"
})

# Obter categorias finais únicas
categorias_finais_natureza = df["NATUREZA"].unique()

# Verificar se há categorias que não sejam 'U' ou 'R'
categorias_invalidas = [cat for cat in categorias_finais_natureza if cat not in ("U", "R")]

# Exibir as categorias finais
print("Categorias finais na coluna NATUREZA:")
print(categorias_finais_natureza)

# Mostrar aviso se existirem categorias inválidas
if categorias_invalidas:
    print("\nAVISO: Foram encontradas as seguintes categorias inesperadas em NATUREZA:")
    print(categorias_invalidas)
else:
    print("\nSem problemas com U ou R.")


Categorias finais na coluna NATUREZA:
['U' 'R']

Sem problemas com U ou R.


In [9]:
# Cole a lista completa (copie e cole todos os dados) dentro da string abaixo:
data = """CODIGO_IBGE_7	MUNICIPIO
4300034	Aceguá
4300059	Água Santa
4300109	Agudo
4300208	Ajuricaba
4300307	Alecrim
4300406	Alegrete
4300455	Alegria
4300471	Almirante Tamandaré Do Sul
4300505	Alpestre
4300554	Alto Alegre
4300570	Alto Feliz
4300604	Alvorada
4300638	Amaral Ferrador
4300646	Ametista Do Sul
4300661	André Da Rocha
4300703	Anta Gorda
4300802	Antônio Prado
4300851	Arambaré
4300877	Araricá
4300901	Aratiba
4301008	Arroio Do Meio
4301057	Arroio Do Sal
4301073	Arroio Do Padre
4301107	Arroio Dos Ratos
4301206	Arroio Do Tigre
4301305	Arroio Grande
4301404	Arvorezinha
4301503	Augusto Pestana
4301552	Áurea
4301602	Bagé
4301636	Balneário Pinhal
4301651	Barão
4301701	Barão De Cotegipe
4301750	Barão Do Triunfo
4301800	Barracão
4301859	Barra Do Guarita
4301875	Barra Do Quaraí
4301909	Barra Do Ribeiro
4301925	Barra Do Rio Azul
4301958	Barra Funda
4302006	Barros Cassal
4302055	Benjamin Constant Do Sul
4302105	Bento Gonçalves
4302154	Boa Vista Das Missões
4302204	Boa Vista Do Buricá
4302220	Boa Vista Do Cadeado
4302238	Boa Vista Do Incra
4302253	Boa Vista Do Sul
4302303	Bom Jesus
4302352	Bom Princípio
4302378	Bom Progresso
4302402	Bom Retiro Do Sul
4302451	Boqueirão Do Leão
4302501	Bossoroca
4302584	Bozano
4302600	Braga
4302659	Brochier
4302709	Butiá
4302808	Caçapava Do Sul
4302907	Cacequi
4303004	Cachoeira Do Sul
4303103	Cachoeirinha
4303202	Cacique Doble
4303301	Caibaté
4303400	Caiçara
4303509	Camaquã
4303558	Camargo
4303608	Cambará Do Sul
4303673	Campestre Da Serra
4303707	Campina Das Missões
4303806	Campinas Do Sul
4303905	Campo Bom
4304002	Campo Novo
4304101	Campos Borges
4304200	Candelária
4304309	Cândido Godói
4304358	Candiota
4304408	Canela
4304507	Canguçu
4304606	Canoas
4304614	Canudos Do Vale
4304622	Capão Bonito Do Sul
4304630	Capão Da Canoa
4304655	Capão Do Cipó
4304663	Capão Do Leão
4304671	Capivari Do Sul
4304689	Capela De Santana
4304697	Capitão
4304705	Carazinho
4304713	Caraá
4304804	Carlos Barbosa
4304853	Carlos Gomes
4304903	Casca
4304952	Caseiros
4305009	Catuípe
4305108	Caxias Do Sul
4305116	Centenário
4305124	Cerrito
4305132	Cerro Branco
4305157	Cerro Grande
4305173	Cerro Grande Do Sul
4305207	Cerro Largo
4305306	Chapada
4305355	Charqueadas
4305371	Charrua
4305405	Chiapetta
4305439	Chuí
4305447	Chuvisca
4305454	Cidreira
4305504	Ciríaco
4305587	Colinas
4305603	Colorado
4305702	Condor
4305801	Constantina
4305835	Coqueiro Baixo
4305850	Coqueiros Do Sul
4305871	Coronel Barros
4305900	Coronel Bicaco
4305934	Coronel Pilar
4305959	Cotiporã
4305975	Coxilha
4306007	Crissiumal
4306056	Cristal
4306072	Cristal Do Sul
4306106	Cruz Alta
4306130	Cruzaltense
4306205	Cruzeiro Do Sul
4306304	David Canabarro
4306320	Derrubadas
4306353	Dezesseis De Novembro
4306379	Dilermando De Aguiar
4306403	Dois Irmãos
4306429	Dois Irmãos Das Missões
4306452	Dois Lajeados
4306502	Dom Feliciano
4306551	Dom Pedro De Alcântara
4306601	Dom Pedrito
4306700	Dona Francisca
4306734	Doutor Maurício Cardoso
4306759	Doutor Ricardo
4306767	Eldorado Do Sul
4306809	Encantado
4306908	Encruzilhada Do Sul
4306924	Engenho Velho
4306932	Entre-Ijuís
4306957	Entre Rios Do Sul
4306973	Erebango
4307005	Erechim
4307054	Ernestina
4307104	Herval
4307203	Erval Grande
4307302	Erval Seco
4307401	Esmeralda
4307450	Esperança Do Sul
4307500	Espumoso
4307559	Estação
4307609	Estância Velha
4307708	Esteio
4307807	Estrela
4307815	Estrela Velha
4307831	Eugênio De Castro
4307864	Fagundes Varela
4307906	Farroupilha
4308003	Faxinal Do Soturno
4308052	Faxinalzinho
4308078	Fazenda Vilanova
4308102	Feliz
4308201	Flores Da Cunha
4308250	Floriano Peixoto
4308300	Fontoura Xavier
4308409	Formigueiro
4308433	Forquetinha
4308458	Fortaleza Dos Valos
4308508	Frederico Westphalen
4308607	Garibaldi
4308656	Garruchos
4308706	Gaurama
4308805	General Câmara
4308854	Gentil
4308904	Getúlio Vargas
4309001	Giruá
4309050	Glorinha
4309100	Gramado
4309126	Gramado Dos Loureiros
4309159	Gramado Xavier
4309209	Gravataí
4309258	Guabiju
4309308	Guaíba
4309407	Guaporé
4309506	Guarani Das Missões
4309555	Harmonia
4309571	Herveiras
4309605	Horizontina
4309654	Hulha Negra
4309704	Humaitá
4309753	Ibarama
4309803	Ibiaçá
4309902	Ibiraiaras
4309951	Ibirapuitã
4310009	Ibirubá
4310108	Igrejinha
4310207	Ijuí
4310306	Ilópolis
4310330	Imbé
4310363	Imigrante
4310405	Independência
4310413	Inhacorá
4310439	Ipê
4310462	Ipiranga Do Sul
4310504	Iraí
4310538	Itaara
4310553	Itacurubi
4310579	Itapuca
4310603	Itaqui
4310652	Itati
4310702	Itatiba Do Sul
4310751	Ivorá
4310801	Ivoti
4310850	Jaboticaba
4310876	Jacuizinho
4310900	Jacutinga
4311007	Jaguarão
4311106	Jaguari
4311122	Jaquirana
4311130	Jari
4311155	Jóia
4311205	Júlio De Castilhos
4311239	Lagoa Bonita Do Sul
4311254	Lagoão
4311270	Lagoa Dos Três Cantos
4311304	Lagoa Vermelha
4311403	Lajeado
4311429	Lajeado Do Bugre
4311502	Lavras Do Sul
4311601	Liberato Salzano
4311627	Lindolfo Collor
4311643	Linha Nova
4311700	Machadinho
4311718	Maçambará
4311734	Mampituba
4311759	Manoel Viana
4311775	Maquiné
4311791	Maratá
4311809	Marau
4311908	Marcelino Ramos
4311981	Mariana Pimentel
4312005	Mariano Moro
4312054	Marques De Souza
4312104	Mata
4312138	Mato Castelhano
4312153	Mato Leitão
4312179	Mato Queimado
4312203	Maximiliano De Almeida
4312252	Minas Do Leão
4312302	Miraguaí
4312351	Montauri
4312377	Monte Alegre Dos Campos
4312385	Monte Belo Do Sul
4312401	Montenegro
4312427	Mormaço
4312443	Morrinhos Do Sul
4312450	Morro Redondo
4312476	Morro Reuter
4312500	Mostardas
4312609	Muçum
4312617	Muitos Capões
4312625	Muliterno
4312658	Não-Me-Toque
4312674	Nicolau Vergueiro
4312708	Nonoai
4312757	Nova Alvorada
4312807	Nova Araçá
4312906	Nova Bassano
4312955	Nova Boa Vista
4313003	Nova Bréscia
4313011	Nova Candelária
4313037	Nova Esperança Do Sul
4313060	Nova Hartz
4313086	Nova Pádua
4313102	Nova Palma
4313201	Nova Petrópolis
4313300	Nova Prata
4313334	Nova Ramada
4313359	Nova Roma Do Sul
4313375	Nova Santa Rita
4313391	Novo Cabrais
4313409	Novo Hamburgo
4313425	Novo Machado
4313441	Novo Tiradentes
4313466	Novo Xingu
4313490	Novo Barreiro
4313508	Osório
4313607	Paim Filho
4313656	Palmares Do Sul
4313706	Palmeira Das Missões
4313805	Palmitinho
4313904	Panambi
4313953	Pantano Grande
4314001	Paraí
4314027	Paraíso Do Sul
4314035	Pareci Novo
4314050	Parobé
4314068	Passa Sete
4314076	Passo Do Sobrado
4314100	Passo Fundo
4314134	Paulo Bento
4314159	Paverama
4314175	Pedras Altas
4314209	Pedro Osório
4314308	Pejuçara
4314407	Pelotas
4314423	Picada Café
4314456	Pinhal
4314464	Pinhal Da Serra
4314472	Pinhal Grande
4314498	Pinheirinho Do Vale
4314506	Pinheiro Machado
4314555	Pirapó
4314605	Piratini
4314704	Planalto
4314753	Poço Das Antas
4314779	Pontão
4314787	Ponte Preta
4314803	Portão
4314902	Porto Alegre
4315008	Porto Lucena
4315057	Porto Mauá
4315073	Porto Vera Cruz
4315107	Porto Xavier
4315131	Pouso Novo
4315149	Presidente Lucena
4315156	Progresso
4315172	Protásio Alves
4315206	Putinga
4315305	Quaraí
4315313	Quatro Irmãos
4315321	Quevedos
4315354	Quinze De Novembro
4315404	Redentora
4315453	Relvado
4315503	Restinga Seca
4315552	Rio Dos Índios
4315602	Rio Grande
4315701	Rio Pardo
4315750	Riozinho
4315800	Roca Sales
4315909	Rodeio Bonito
4315958	Rolador
4316006	Rolante
4316105	Ronda Alta
4316204	Rondinha
4316303	Roque Gonzales
4316402	Rosário Do Sul
4316428	Sagrada Família
4316436	Saldanha Marinho
4316451	Salto Do Jacuí
4316477	Salvador Das Missões
4316501	Salvador Do Sul
4316600	Sananduva
4316709	Santa Bárbara Do Sul
4316733	Santa Cecília Do Sul
4316758	Santa Clara Do Sul
4316808	Santa Cruz Do Sul
4316907	Santa Maria
4316956	Santa Maria Do Herval
4316972	Santa Margarida Do Sul
4317004	Santana Da Boa Vista
4317103	Sant'Ana Do Livramento
4317202	Santa Rosa
4317251	Santa Tereza
4317301	Santa Vitória Do Palmar
4317400	Santiago
4317509	Santo Ângelo
4317558	Santo Antônio Do Palma
4317608	Santo Antônio Da Patrulha
4317707	Santo Antônio Das Missões
4317756	Santo Antônio Do Planalto
4317806	Santo Augusto
4317905	Santo Cristo
4317954	Santo Expedito Do Sul
4318002	São Borja
4318051	São Domingos Do Sul
4318101	São Francisco De Assis
4318200	São Francisco De Paula
4318309	São Gabriel
4318408	São Jerônimo
4318424	São João Da Urtiga
4318432	São João Do Polêsine
4318440	São Jorge
4318457	São José Das Missões
4318465	São José Do Herval
4318481	São José Do Hortêncio
4318499	São José Do Inhacorá
4318507	São José Do Norte
4318606	São José Do Ouro
4318614	São José Do Sul
4318622	São José Dos Ausentes
4318705	São Leopoldo
4318804	São Lourenço Do Sul
4318903	São Luiz Gonzaga
4319000	São Marcos
4319109	São Martinho
4319125	São Martinho Da Serra
4319158	São Miguel Das Missões
4319208	São Nicolau
4319307	São Paulo Das Missões
4319356	São Pedro Da Serra
4319364	São Pedro Das Missões
4319372	São Pedro Do Butiá
4319406	São Pedro Do Sul
4319505	São Sebastião Do Caí
4319604	São Sepé
4319703	São Valentim
4319711	São Valentim Do Sul
4319737	São Valério Do Sul
4319752	São Vendelino
4319802	São Vicente Do Sul
4319901	Sapiranga
4320008	Sapucaia Do Sul
4320107	Sarandi
4320206	Seberi
4320230	Sede Nova
4320263	Segredo
4320305	Selbach
4320321	Senador Salgado Filho
4320354	Sentinela Do Sul
4320404	Serafina Corrêa
4320453	Sério
4320503	Sertão
4320552	Sertão Santana
4320578	Sete De Setembro
4320602	Severiano De Almeida
4320651	Silveira Martins
4320677	Sinimbu
4320701	Sobradinho
4320800	Soledade
4320859	Tabaí
4320909	Tapejara
4321006	Tapera
4321105	Tapes
4321204	Taquara
4321303	Taquari
4321329	Taquaruçu Do Sul
4321352	Tavares
4321402	Tenente Portela
4321436	Terra De Areia
4321451	Teutônia
4321469	Tio Hugo
4321477	Tiradentes Do Sul
4321493	Toropi
4321501	Torres
4321600	Tramandaí
4321626	Travesseiro
4321634	Três Arroios
4321667	Três Cachoeiras
4321709	Três Coroas
4321808	Três De Maio
4321832	Três Forquilhas
4321857	Três Palmeiras
4321907	Três Passos
4321956	Trindade Do Sul
4322004	Triunfo
4322103	Tucunduva
4322152	Tunas
4322186	Tupanci Do Sul
4322202	Tupanciretã
4322251	Tupandi
4322301	Tuparendi
4322327	Turuçu
4322343	Ubiretama
4322350	União Da Serra
4322376	Unistalda
4322400	Uruguaiana
4322509	Vacaria
4322525	Vale Verde
4322533	Vale Do Sol
4322541	Vale Real
4322558	Vanini
4322608	Venâncio Aires
4322707	Vera Cruz
4322806	Veranópolis
4322855	Vespasiano Correa
4322905	Viadutos
4323002	Viamão
4323101	Vicente Dutra
4323200	Victor Graeff
4323309	Vila Flores
4323358	Vila Lângaro
4323408	Vila Maria
4323457	Vila Nova Do Sul
4323507	Vista Alegre
4323606	Vista Alegre Do Prata
4323705	Vista Gaúcha
4323754	Vitória Das Missões
4323770	Westfalia
4323804	Xangri-Lá
4314548	Pinto Bandeira
"""

# Cria o DataFrame usando o separador de tabulação
df2 = pd.read_csv(StringIO(data), sep="\t")
df5=df2
# Exibe as primeiras linhas do DataFrame
df2.head()


,CODIGO_IBGE_7,MUNICIPIO
0,4300034,Aceguá
1,4300059,Água Santa
2,4300109,Agudo
3,4300208,Ajuricaba
4,4300307,Alecrim


In [10]:
df4 = df

In [11]:
# Função para limpar os nomes dos municípios
def limpar_municipio(nome):
    nome = unidecode.unidecode(nome)  # Remove acentos
    nome = re.sub(r"[^A-Z0-9]", "", nome.upper())  # Remove caracteres especiais, mantendo apenas letras e números
    return nome

# Aplicar a função na coluna 'municipio'
df2["MUNICIPIO"] = df2["MUNICIPIO"].apply(limpar_municipio)



In [12]:
# Fazer o merge do df com df2 baseado na coluna 'municipio'
df = df.merge(df2, on="MUNICIPIO", how="left")


In [13]:
# Exibir as informações do DataFrame
df.info()

# Verificar se há valores nulos
colunas_nulas = df.columns[df.isnull().any()].tolist()

# Exibir mensagem personalizada
if not colunas_nulas:
    print("\n✅ Nenhuma das colunas tem valores nulos.")
else:
    print("\n⚠️ As seguintes colunas possuem valores nulos:", ", ".join(colunas_nulas))

# --- Checar existência de municípios específicos ---
# --- Checar existência de municípios específicos ---
municipios_a_checar = [
    "AGUDO",
    "ANTONIOPRADO",
    "ARROIODOTIGRE",
    "ARROIOGRANDE",
    "ARVOREZINHA",
    "BARRADORIOAZUL",
    "BENTOGONCALVES",
    "CAICARA",
    "CANOAS",
    "CANUDOSDOVALE",
    "CERROBRANCO",
    "CHARQUEADAS",
    "ELDORADODOSUL",
    "ENCANTADO",
    "ENCRUZILHADADOSUL",
    "ESPUMOSO",
    "ESTEIO",
    "FAXINALDOSOTURNO",
    "FELIZ",
    "GUAPORE",
    "LIBERATOSALZANO",
    "NOVAPALMA",
    "NOVASANTARITA",
    "PORTOALEGRE",
    "RELVADO",
    "RESTINGASECA",
    "ROCASALES",
    "SAOFRANCISCODEPAULA",
    "SAOLEOPOLDO",
    "SOBRADINHO",
    "TAPES",
    "TAQUARI",
    "VERACRUZ",
    "VICENTEDUTRA"
]  # fecha aqui!

for mun in municipios_a_checar:
    if mun in df["MUNICIPIO"].values:
        print(f"✅ {mun} existe na coluna MUNICIPIO.")
    else:
        print(f"❌ {mun} NÃO encontrado na coluna MUNICIPIO - Demanda Larissa.")



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 381 entries, 0 to 380
Data columns (total 13 columns):
 #   Column                                    Non-Null Count  Dtype 
---  ------                                    --------------  ----- 
 0   ITEM                                      381 non-null    object
 1   NOME                                      381 non-null    object
 2   CPF                                       381 non-null    object
 3   NIS                                       381 non-null    object
 4   ENDERECO                                  381 non-null    object
 5   BAIRRO                                    381 non-null    object
 6   LATITUDE                                  381 non-null    object
 7   LONGITUDE                                 381 non-null    object
 8   SITUACAO DA HABITACAO                     381 non-null    object
 9   CONCLUSAO DO LAUDO TECNICO DE ENGENHARIA  381 non-null    object
 10  NATUREZA                                  381 non-

In [14]:
#quantidade_zeros = (df['CPF'] == '00000000000').sum()

#print(f"Existem {quantidade_zeros} registros com CPF '00000000000'.")


# Supondo que você já tenha carregado o DataFrame `df`

colunas = ['CPF', 'NOME', 'ENDERECO', 'BAIRRO']

for col in colunas:
    # Padroniza para string, remove NaN e espaços em branco
    series = df[col].fillna('').astype(str).str.strip()
    print(f"Coluna `{col}`:")

    # Função auxiliar para formatar o breakdown por município
    def breakdown(mask):
        vc = df.loc[mask, 'MUNICIPIO'].value_counts()
        return ' e '.join(f"{cnt} registros no município {muni}" for muni, cnt in vc.items())

    if col == 'CPF':
        # '00000000000'
        mask_11zeros = series == '00000000000'
        q_11zeros = mask_11zeros.sum()
        if q_11zeros:
            print(f"  - {q_11zeros} registros com '00000000000', sendo {breakdown(mask_11zeros)}")
        else:
            print(f"  - 0 registros com '00000000000'")

        # '0'
        mask_zero = series == '0'
        q_zero = mask_zero.sum()
        if q_zero:
            print(f"  - {q_zero} registros com '0', sendo {breakdown(mask_zero)}")
        else:
            print(f"  - 0 registros com '0'")

        # vazios
        mask_vazio = series == ''
        q_vazios = mask_vazio.sum()
        if q_vazios:
            print(f"  - {q_vazios} registros vazios, sendo {breakdown(mask_vazio)}")
        else:
            print(f"  - 0 registros vazios")

    else:
        # '0'
        mask_zero = series == '0'
        q_zero = mask_zero.sum()
        if q_zero:
            print(f"  - {q_zero} registros com '0', sendo {breakdown(mask_zero)}")
        else:
            print(f"  - 0 registros com '0'")

        # vazios
        mask_vazio = series == ''
        q_vazios = mask_vazio.sum()
        if q_vazios:
            print(f"  - {q_vazios} registros vazios, sendo {breakdown(mask_vazio)}")
        else:
            print(f"  - 0 registros vazios")

    print()  # quebra de linha entre colunas


Coluna `CPF`:
  - 103 registros com '00000000000', sendo 80 registros no município PORTOALEGRE e 16 registros no município SAOJERONIMO e 5 registros no município CANOAS e 1 registros no município SAOMARTINHODASERRA e 1 registros no município BARAO
  - 0 registros com '0'
  - 0 registros vazios

Coluna `NOME`:
  - 0 registros com '0'
  - 0 registros vazios

Coluna `ENDERECO`:
  - 0 registros com '0'
  - 3 registros vazios, sendo 3 registros no município SAOMARTINHODASERRA

Coluna `BAIRRO`:
  - 0 registros com '0'
  - 6 registros vazios, sendo 5 registros no município SAOJERONIMO e 1 registros no município SAOMARTINHODASERRA



In [15]:
import os
import pandas as pd
from datetime import datetime
from google.colab import files

# ===== Caminho base no Colab =====
caminho_saida = "/content/normal"
os.makedirs(caminho_saida, exist_ok=True)

# Criar listas para os relatórios
relatorio_municipios = []
totais_por_municipio = []

# Criar e salvar um CSV para cada município
for municipio, grupo in df.groupby("MUNICIPIO"):
    cod_ibge = grupo["CODIGO_IBGE_7"].iloc[0]

    # Relatório por região
    contagem_regiao = grupo["NATUREZA"].value_counts()
    for regiao, num_linhas in contagem_regiao.items():
        nome_regiao = "urbano" if regiao == "U" else "rural"
        relatorio_municipios.append([municipio, cod_ibge, num_linhas, nome_regiao])

    # Relatório de totais por município
    total_municipio = len(grupo)
    totais_por_municipio.append([municipio, cod_ibge, total_municipio])

    # Gera a data de hoje no formato yyyymmdd
    data_de_hoje = datetime.today().strftime("%Y%m%d")

    # Criar nome do arquivo CSV individual
    nome_arquivo = f"C.ACA.WWW.318.{data_de_hoje}.{municipio}.{cod_ibge}.SNH.{numero_lote}.csv"
    caminho_arquivo = os.path.join(caminho_saida, nome_arquivo)

    # Preparar o grupo para salvar
    grupo_csv = grupo.drop(columns=["MUNICIPIO", "CODIGO_IBGE_7"]).copy()
    grupo_csv["CPF"] = grupo_csv["CPF"].astype(str)
    grupo_csv["NIS"] = grupo_csv["NIS"].astype(str)

    colunas_cpf_membros = [
        "GF_CPF_MEMBRO_01", "GF_CPF_MEMBRO_02", "GF_CPF_MEMBRO_03", "GF_CPF_MEMBRO_04",
        "GF_CPF_MEMBRO_05", "GF_CPF_MEMBRO_06", "GF_CPF_MEMBRO_07", "GF_CPF_MEMBRO_08",
        "GF_CPF_MEMBRO_09", "GF_CPF_MEMBRO_10"
    ]
    for coluna in colunas_cpf_membros:
        if coluna in grupo_csv.columns:
            grupo_csv[coluna] = grupo_csv[coluna].astype(str)

    grupo_csv = grupo_csv.applymap(
        lambda x: str(x).replace('\n', ' ').replace('\r', ' ') if isinstance(x, str) else x
    )

    grupo_csv.to_csv(caminho_arquivo, index=False, encoding="utf-8", sep=";", quoting=3)

# Criar DataFrames para os dois relatórios
df_regiao = pd.DataFrame(relatorio_municipios, columns=["MUNICIPIO", "CODIGO_IBGE_7", "NUMERO_DE_LINHAS", "NATUREZA"])
df_totais = pd.DataFrame(totais_por_municipio, columns=["MUNICIPIO", "CODIGO_IBGE_7", "TOTAL_LINHAS"])

# Adicionar linha de total geral
linha_total = pd.DataFrame({
    "MUNICIPIO": ["TOTAL GERAL"],
    "CODIGO_IBGE_7": [""],
    "TOTAL_LINHAS": [df_totais["TOTAL_LINHAS"].sum()]
})
df_totais = pd.concat([df_totais, linha_total], ignore_index=True)

# Salvar relatório Excel
arquivo_relatorio = os.path.join(caminho_saida, "Relatorio_Municipios_normais.xlsx")
with pd.ExcelWriter(arquivo_relatorio, engine="openpyxl") as writer:
    df_regiao.to_excel(writer, sheet_name="Região", index=False)
    df_totais.to_excel(writer, sheet_name="Total por Município", index=False)

print("✅ Arquivos gerados em:", caminho_saida)

# ====== Baixar todos os arquivos ======
# Baixar Excel final
files.download(arquivo_relatorio)

# Baixar cada CSV gerado
for nome_arquivo in os.listdir(caminho_saida):
    if nome_arquivo.endswith(".csv"):
        files.download(os.path.join(caminho_saida, nome_arquivo))

/tmp/ipython-input-375659813.py:49: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  grupo_csv = grupo_csv.applymap(
/tmp/ipython-input-375659813.py:49: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  grupo_csv = grupo_csv.applymap(
/tmp/ipython-input-375659813.py:49: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  grupo_csv = grupo_csv.applymap(
/tmp/ipython-input-375659813.py:49: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  grupo_csv = grupo_csv.applymap(
/tmp/ipython-input-375659813.py:49: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  grupo_csv = grupo_csv.applymap(
/tmp/ipython-input-375659813.py:49: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  grupo_csv = grupo_csv.applymap(
/tmp/ipython-input-375659813.py:49: FutureWarning: DataFrame.applymap has been deprecate

✅ Arquivos gerados em: /content/normal


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#####